# von Mises Mixture Model on the Torus

In `torus_kmeans.ipynb`, we saw how standard K-means fails on the torus because Euclidean distance doesn't respect wraparound. We fixed this by using **toroidal distance** (shortest arc in each dimension) and **circular means** (atan2-based averaging). But K-means still has fundamental limitations:

- **Hard assignments** — every point belongs to exactly one cluster
- **No notion of spread** — all clusters are treated as having the same "shape"
- **No probabilistic interpretation** — we can't ask "how likely is this new point?"

### Why Gaussian Doesn't Work on a Torus

On the real line, the Gaussian $\mathcal{N}(\mu, \sigma^2)$ is the natural choice for soft clustering. But on a circle (or torus), angles **wrap around**: 359° and 1° are only 2° apart, yet a Gaussian centered at 0° would assign 359° an extremely low density. The Gaussian has no concept of periodicity.

### The von Mises Distribution: "Gaussian on a Circle"

The **von Mises distribution** is the circular analogue of the Gaussian:

$$p(\theta \mid \mu, \kappa) = \frac{\exp(\kappa \cos(\theta - \mu))}{2\pi I_0(\kappa)}$$

where:
- $\mu \in [0, 2\pi)$ is the **mean direction**
- $\kappa \geq 0$ is the **concentration** parameter (analogous to $1/\sigma^2$ — higher $\kappa$ means tighter)
- $I_0(\kappa)$ is the modified Bessel function of the first kind, order 0

The key insight: $\cos(\theta - \mu)$ is **naturally periodic**, so the density wraps around seamlessly!

### Product of von Mises for the 2D Torus

For our 2D torus $[0, 2\pi) \times [0, 2\pi)$, we use a **product of independent von Mises** distributions, one per angular dimension:

$$p(\theta, \phi \mid \mu_\theta, \mu_\phi, \kappa_\theta, \kappa_\phi) = p(\theta \mid \mu_\theta, \kappa_\theta) \cdot p(\phi \mid \mu_\phi, \kappa_\phi)$$

This gives us:
- **Soft assignments** via posterior probabilities
- **Different tightness per cluster per dimension** ($\kappa_{k\theta}$ and $\kappa_{k\phi}$ can differ)
- A proper **generative model** we can sample from

## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.special import i0e  # exponentially-scaled I_0 Bessel function

# For nice inline plots
plt.rcParams['figure.dpi'] = 100

# Colors matching our series style
COLORS = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

print('Setup complete!')

## 2. Generate Pre-Clustered Points on a Flat Torus

We reuse the same approach from `torus_kmeans.ipynb`: 5 clusters on the flat torus $[0, 2\pi) \times [0, 2\pi)$, with some clusters deliberately straddling the wraparound boundary to test our algorithm.

In [ ]:
def wrap_to_period(x, period=2*np.pi):
    """Wrap values to [0, period)."""
    return x % period

# Set seed for reproducible data generation
np.random.seed(42)

# True cluster parameters: (center_theta, center_phi, kappa_theta, kappa_phi, n_points)
true_params = [
    (1.0, 1.0, 8.0, 12.0, 60),     # Cluster 0: tight, interior
    (4.5, 4.8, 6.0, 6.0, 60),      # Cluster 1: medium, interior
    (0.1, 3.5, 10.0, 5.0, 60),     # Cluster 2: near θ=0 boundary!
    (3.0, 0.05, 7.0, 9.0, 60),     # Cluster 3: near φ=0 boundary!
    (5.8, 5.9, 5.0, 5.0, 60),      # Cluster 4: near BOTH boundaries (corner)!
]

K_true = len(true_params)
data_list = []
labels_true = []

for k, (mu_t, mu_p, kap_t, kap_p, n) in enumerate(true_params):
    # Sample from von Mises in each dimension
    theta_samples = np.random.vonmises(mu_t, kap_t, size=n)
    phi_samples = np.random.vonmises(mu_p, kap_p, size=n)
    # vonmises returns in [-pi, pi), wrap to [0, 2pi)
    theta_samples = wrap_to_period(theta_samples)
    phi_samples = wrap_to_period(phi_samples)
    data_list.append(np.column_stack([theta_samples, phi_samples]))
    labels_true.extend([k] * n)

data = np.vstack(data_list)  # shape (N, 2)
labels_true = np.array(labels_true)
N = len(data)

print(f'Generated {N} points in {K_true} clusters on the torus [0, 2π) × [0, 2π)')
print(f'Points per cluster: {[p[4] for p in true_params]}')

# Quick visualization
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
for k in range(K_true):
    mask = labels_true == k
    ax.scatter(data[mask, 0], data[mask, 1], c=COLORS[k], s=20, alpha=0.6,
              label=f'Cluster {k}')
    ax.scatter(true_params[k][0], true_params[k][1], c=COLORS[k], s=200,
              marker='*', edgecolors='black', linewidths=1, zorder=5)

ax.set_xlim(0, 2*np.pi)
ax.set_ylim(0, 2*np.pi)
ax.set_xlabel(r'$\theta$', fontsize=14)
ax.set_ylabel(r'$\phi$', fontsize=14)
ax.set_title('Ground Truth Clusters on Flat Torus', fontsize=16)
ax.legend(fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. The von Mises Distribution

Before running EM, let's build intuition for the von Mises distribution.

$$p(\theta \mid \mu, \kappa) = \frac{\exp(\kappa \cos(\theta - \mu))}{2\pi I_0(\kappa)}$$

Key properties:
- **$\kappa = 0$**: uniform distribution on the circle (no preferred direction)
- **$\kappa \to \infty$**: all mass concentrated at $\mu$ (like a Dirac delta)
- **Moderate $\kappa$** (2–10): bell-shaped, centered at $\mu$, wrapping around naturally

The normalizing constant $I_0(\kappa)$ ensures the density integrates to 1 over $[0, 2\pi)$.

In [ ]:
def von_mises_pdf(theta, mu, kappa):
    """Compute von Mises PDF using exponentially-scaled Bessel function for stability.
    
    Uses: p(θ|μ,κ) = exp(κ cos(θ-μ)) / (2π I_0(κ))
    Rewritten as: exp(κ cos(θ-μ) - κ) / (2π i0e(κ))
    where i0e(κ) = I_0(κ) * exp(-κ)
    """
    return np.exp(kappa * (np.cos(theta - mu) - 1)) / (2 * np.pi * i0e(kappa))

# Visualize von Mises for different κ values
theta_grid = np.linspace(0, 2*np.pi, 500)
mu_demo = np.pi  # center at π for clear visualization
kappas = [0.5, 2, 5, 10, 30]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: Cartesian plot
ax = axes[0]
for i, kap in enumerate(kappas):
    pdf_vals = von_mises_pdf(theta_grid, mu_demo, kap)
    ax.plot(theta_grid, pdf_vals, color=COLORS[i % len(COLORS)], linewidth=2,
            label=rf'$\kappa = {kap}$')
ax.axvline(mu_demo, color='gray', linestyle='--', alpha=0.5, label=rf'$\mu = \pi$')
ax.set_xlabel(r'$\theta$', fontsize=14)
ax.set_ylabel(r'$p(\theta)$', fontsize=14)
ax.set_title('von Mises PDF for Various Concentrations', fontsize=15)
ax.legend(fontsize=12)
ax.set_xlim(0, 2*np.pi)
ax.grid(True, alpha=0.3)

# Right: Polar plot (showing the circular nature)
ax2 = plt.subplot(122, projection='polar')
for i, kap in enumerate(kappas):
    pdf_vals = von_mises_pdf(theta_grid, mu_demo, kap)
    ax2.plot(theta_grid, pdf_vals, color=COLORS[i % len(COLORS)], linewidth=2,
             label=rf'$\kappa = {kap}$')
ax2.set_title('Polar View (Circular Density)', fontsize=15, pad=20)
ax2.legend(fontsize=10, loc='lower right', bbox_to_anchor=(1.3, 0.0))

plt.tight_layout()
plt.show()

print('Notice how the density wraps seamlessly around the circle!')
print('At κ=0.5, it\'s nearly uniform. At κ=30, it\'s sharply concentrated at μ.')

## 4. EM Algorithm for Toroidal von Mises Mixture

Our mixture model on the 2D torus is:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \prod_{d=1}^{2} \frac{\exp(\kappa_{kd} \cos(x_d - \mu_{kd}))}{2\pi I_0(\kappa_{kd})}$$

### E-step

Compute responsibilities (posterior probability that point $n$ belongs to cluster $k$):

$$r_{nk} = \frac{\pi_k \prod_d \text{vM}(x_{nd} \mid \mu_{kd}, \kappa_{kd})}{\sum_j \pi_j \prod_d \text{vM}(x_{nd} \mid \mu_{jd}, \kappa_{jd})}$$

We work in **log space** for stability:
$$\log p_k(\mathbf{x}_n) = \sum_d \left[\kappa_{kd} \cos(x_{nd} - \mu_{kd}) - \log(2\pi) - \log I_0(\kappa_{kd})\right]$$

### M-step

**Circular mean** (weighted by responsibilities):
$$\bar{C}_{kd} = \sum_n r_{nk} \cos(x_{nd}), \quad \bar{S}_{kd} = \sum_n r_{nk} \sin(x_{nd})$$
$$\mu_{kd} = \text{atan2}(\bar{S}_{kd}, \bar{C}_{kd})$$

**Concentration** (Mardia-Jupp approximation):
$$\bar{R}_{kd} = \frac{\sqrt{\bar{C}_{kd}^2 + \bar{S}_{kd}^2}}{N_k}, \quad \kappa_{kd} \approx \frac{\bar{R}_{kd}(2 - \bar{R}_{kd}^2)}{1 - \bar{R}_{kd}^2}$$

**Mixing weights**: $\pi_k = N_k / N$ where $N_k = \sum_n r_{nk}$

In [ ]:
def log_von_mises_pdf(x, mu, kappa):
    """Log of von Mises PDF, using i0e for numerical stability.
    
    log p(x|mu,kappa) = kappa * cos(x - mu) - log(2*pi) - log(I_0(kappa))
    
    Since i0e(kappa) = I_0(kappa) * exp(-kappa):
        log(I_0(kappa)) = log(i0e(kappa)) + kappa
    
    So: log p = kappa * cos(x - mu) - log(2*pi) - log(i0e(kappa)) - kappa
             = kappa * (cos(x - mu) - 1) - log(2*pi) - log(i0e(kappa))
    """
    return kappa * (np.cos(x - mu) - 1) - np.log(2 * np.pi) - np.log(i0e(kappa))


def log_sum_exp(log_vals, axis=None):
    """Numerically stable log-sum-exp."""
    max_val = np.max(log_vals, axis=axis, keepdims=True)
    return max_val.squeeze(axis=axis) + np.log(np.sum(np.exp(log_vals - max_val), axis=axis))


def estimate_kappa(Rbar):
    """Mardia-Jupp approximation for von Mises concentration.
    
    Given mean resultant length R_bar, approximate kappa such that
    A(kappa) = I_1(kappa)/I_0(kappa) = R_bar.
    
    Uses: kappa ≈ R_bar * (2 - R_bar^2) / (1 - R_bar^2)
    """
    Rbar = np.clip(Rbar, 1e-10, 1 - 1e-10)
    return Rbar * (2 - Rbar**2) / (1 - Rbar**2)


def toroidal_vm_em(data, K, max_iter=200, tol=1e-8, seed=123):
    """EM algorithm for a mixture of product-von-Mises on the 2D torus.
    
    Parameters
    ----------
    data : ndarray, shape (N, 2)
        Angles in [0, 2π) for each dimension.
    K : int
        Number of mixture components.
    max_iter : int
        Maximum EM iterations.
    tol : float
        Convergence threshold on log-likelihood change.
    seed : int
        Random seed for initialization.
    
    Returns
    -------
    mu : ndarray, shape (K, 2) — mean directions
    kappa : ndarray, shape (K, 2) — concentrations
    pi_k : ndarray, shape (K,) — mixing weights
    responsibilities : ndarray, shape (N, K) — final soft assignments
    log_likelihoods : list — log-likelihood at each iteration
    """
    rng = np.random.RandomState(seed)
    N, D = data.shape  # D = 2 for torus
    
    # --- Initialization ---
    # Random initialization: pick K random data points as initial means
    init_idx = rng.choice(N, K, replace=False)
    mu = data[init_idx].copy()  # (K, 2)
    kappa = np.full((K, D), 5.0)  # moderate initial concentration
    pi_k = np.full(K, 1.0 / K)  # uniform mixing weights
    
    log_likelihoods = []
    
    for iteration in range(max_iter):
        # === E-step ===
        # Compute log p(x_n | component k) for each n, k
        # log p_k(x_n) = sum_d [kappa_kd * (cos(x_nd - mu_kd) - 1) - log(2pi) - log(i0e(kappa_kd))]
        log_component = np.zeros((N, K))
        for k in range(K):
            for d in range(D):
                log_component[:, k] += log_von_mises_pdf(data[:, d], mu[k, d], kappa[k, d])
        
        # Add log mixing weights
        log_weighted = log_component + np.log(pi_k)[np.newaxis, :]  # (N, K)
        
        # Log-likelihood
        ll = np.sum(log_sum_exp(log_weighted, axis=1))
        log_likelihoods.append(ll)
        
        # Responsibilities via log-sum-exp
        log_resp = log_weighted - log_sum_exp(log_weighted, axis=1)[:, np.newaxis]
        resp = np.exp(log_resp)  # (N, K)
        
        # Check convergence
        if iteration > 0 and abs(log_likelihoods[-1] - log_likelihoods[-2]) < tol:
            print(f'Converged at iteration {iteration}')
            break
        
        # === M-step ===
        Nk = resp.sum(axis=0)  # (K,) effective number of points per cluster
        Nk = np.maximum(Nk, 1e-10)  # prevent division by zero
        
        # Update mixing weights
        pi_k = Nk / N
        
        # Update means and concentrations per dimension
        for k in range(K):
            for d in range(D):
                # Weighted circular mean
                C_bar = np.sum(resp[:, k] * np.cos(data[:, d]))
                S_bar = np.sum(resp[:, k] * np.sin(data[:, d]))
                mu[k, d] = np.arctan2(S_bar, C_bar) % (2 * np.pi)
                
                # Mean resultant length
                R_bar = np.sqrt(C_bar**2 + S_bar**2) / Nk[k]
                
                # Estimate concentration
                kappa[k, d] = estimate_kappa(R_bar)
    
    else:
        print(f'Reached max iterations ({max_iter})')
    
    return mu, kappa, pi_k, resp, log_likelihoods

print('EM algorithm defined.')

In [ ]:
# Run EM
K = 5
mu_em, kappa_em, pi_em, resp_em, ll_history = toroidal_vm_em(data, K, max_iter=200, seed=123)

# Hard assignments from soft responsibilities
labels_em = np.argmax(resp_em, axis=1)

print(f'\nFinal parameters after {len(ll_history)} iterations:')
print(f'{"Cluster":>8} {"μ_θ":>8} {"μ_φ":>8} {"κ_θ":>8} {"κ_φ":>8} {"π_k":>8}')
print('-' * 52)
for k in range(K):
    print(f'{k:>8d} {mu_em[k,0]:>8.3f} {mu_em[k,1]:>8.3f} '
          f'{kappa_em[k,0]:>8.2f} {kappa_em[k,1]:>8.2f} {pi_em[k]:>8.3f}')

## 5. Visualizations

Let's visualize our results from multiple perspectives.

### 5a. 3D Torus Surface Plot

We embed the flat torus coordinates $(\theta, \phi)$ onto a 3D torus surface using:

$$x = (R + r\cos\phi)\cos\theta, \quad y = (R + r\cos\phi)\sin\theta, \quad z = r\sin\phi$$

In [ ]:
def torus_embed(theta, phi, R_major=3, R_minor=1):
    """Embed flat torus coordinates into 3D."""
    x = (R_major + R_minor * np.cos(phi)) * np.cos(theta)
    y = (R_major + R_minor * np.cos(phi)) * np.sin(theta)
    z = R_minor * np.sin(phi)
    return x, y, z

R_major, R_minor = 3, 1

# Create torus surface mesh
u_grid = np.linspace(0, 2*np.pi, 80)
v_grid = np.linspace(0, 2*np.pi, 80)
U, V = np.meshgrid(u_grid, v_grid)
X_surf, Y_surf, Z_surf = torus_embed(U, V, R_major, R_minor)

# Embed data points
X_pts, Y_pts, Z_pts = torus_embed(data[:, 0], data[:, 1], R_major, R_minor)

# Embed centroids
X_cen, Y_cen, Z_cen = torus_embed(mu_em[:, 0], mu_em[:, 1], R_major, R_minor)

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Semi-transparent torus surface
ax.plot_surface(X_surf, Y_surf, Z_surf, alpha=0.08, color='gray', edgecolor='none')

# Plot points colored by EM assignment
for k in range(K):
    mask = labels_em == k
    ax.scatter(X_pts[mask], Y_pts[mask], Z_pts[mask],
              c=COLORS[k], s=15, alpha=0.6, label=f'Cluster {k}')

# Plot centroids
ax.scatter(X_cen, Y_cen, Z_cen, c='black', s=200, marker='*',
          edgecolors='white', linewidths=1, zorder=10, label='Centroids')

ax.set_title('von Mises Mixture on 3D Torus', fontsize=16)
ax.legend(fontsize=10, loc='upper left')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.view_init(elev=25, azim=45)
plt.tight_layout()
plt.show()

### 5b. 2D Unwrapped Torus (Flat Square) with Posterior Regions

The "Pac-Man view" — the flat torus $[0, 2\pi) \times [0, 2\pi)$ with soft Voronoi regions showing which cluster has the highest posterior probability at each point.

In [ ]:
# Compute posterior probability map over a grid
grid_res = 200
theta_g = np.linspace(0, 2*np.pi, grid_res)
phi_g = np.linspace(0, 2*np.pi, grid_res)
TH, PH = np.meshgrid(theta_g, phi_g)
grid_points = np.column_stack([TH.ravel(), PH.ravel()])  # (grid_res^2, 2)

# Compute log responsibilities for grid points
log_component_grid = np.zeros((grid_res**2, K))
for k in range(K):
    for d in range(2):
        log_component_grid[:, k] += log_von_mises_pdf(grid_points[:, d], mu_em[k, d], kappa_em[k, d])

log_weighted_grid = log_component_grid + np.log(pi_em)[np.newaxis, :]
log_resp_grid = log_weighted_grid - log_sum_exp(log_weighted_grid, axis=1)[:, np.newaxis]
resp_grid = np.exp(log_resp_grid)

# Assignment map: which cluster has highest posterior
assignment_grid = np.argmax(resp_grid, axis=1).reshape(grid_res, grid_res)
confidence_grid = np.max(resp_grid, axis=1).reshape(grid_res, grid_res)

# Build colored background
from matplotlib.colors import ListedColormap, to_rgba

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Background: soft Voronoi with transparency based on confidence
bg_image = np.zeros((grid_res, grid_res, 4))
for k in range(K):
    rgba = np.array(to_rgba(COLORS[k]))
    mask = assignment_grid == k
    bg_image[mask] = rgba
    bg_image[mask, 3] = 0.15 + 0.35 * confidence_grid[mask]  # vary alpha with confidence

ax.imshow(bg_image, extent=[0, 2*np.pi, 0, 2*np.pi], origin='lower', aspect='equal')

# Plot data points
for k in range(K):
    mask = labels_em == k
    ax.scatter(data[mask, 0], data[mask, 1], c=COLORS[k], s=20, alpha=0.7,
              edgecolors='white', linewidths=0.3, label=f'Cluster {k}')

# Plot centroids
for k in range(K):
    ax.scatter(mu_em[k, 0], mu_em[k, 1], c=COLORS[k], s=300, marker='*',
              edgecolors='black', linewidths=1.5, zorder=10)

ax.set_xlim(0, 2*np.pi)
ax.set_ylim(0, 2*np.pi)
ax.set_xlabel(r'$\theta$', fontsize=14)
ax.set_ylabel(r'$\phi$', fontsize=14)
ax.set_title('2D Flat Torus — Soft Voronoi Regions (von Mises Mixture)', fontsize=15)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 5c. Soft Assignment Visualization

Each point's opacity reflects how **confident** the model is about its assignment. Points near cluster boundaries appear more transparent.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plot each point with opacity = max responsibility (confidence)
max_resp = np.max(resp_em, axis=1)  # confidence for each point

for k in range(K):
    mask = labels_em == k
    for i in np.where(mask)[0]:
        ax.scatter(data[i, 0], data[i, 1], c=COLORS[k], s=25,
                  alpha=float(max_resp[i]), edgecolors='none')
    # Dummy for legend
    ax.scatter([], [], c=COLORS[k], s=40, label=f'Cluster {k}')

# Centroids
for k in range(K):
    ax.scatter(mu_em[k, 0], mu_em[k, 1], c=COLORS[k], s=300, marker='*',
              edgecolors='black', linewidths=1.5, zorder=10)

ax.set_xlim(0, 2*np.pi)
ax.set_ylim(0, 2*np.pi)
ax.set_xlabel(r'$\theta$', fontsize=14)
ax.set_ylabel(r'$\phi$', fontsize=14)
ax.set_title('Soft Assignments — Opacity = Confidence', fontsize=15)
ax.legend(fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Report statistics
print(f'Mean confidence: {max_resp.mean():.3f}')
print(f'Min confidence:  {max_resp.min():.3f}')
print(f'Points with confidence > 0.95: {(max_resp > 0.95).sum()} / {N}')

### 5d. Convergence Plot

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(range(len(ll_history)), ll_history, 'o-', color=COLORS[2], markersize=4, linewidth=1.5)
ax.set_xlabel('Iteration', fontsize=13)
ax.set_ylabel('Log-Likelihood', fontsize=13)
ax.set_title('EM Convergence — von Mises Mixture on Torus', fontsize=15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Initial LL: {ll_history[0]:.2f}')
print(f'Final LL:   {ll_history[-1]:.2f}')
print(f'Improvement: {ll_history[-1] - ll_history[0]:.2f}')

### 5e. Per-Dimension Concentration Comparison

Compare the estimated $\kappa$ values to the true values used to generate the data.

In [ ]:
# Match estimated clusters to true clusters by finding best assignment
# (EM doesn't preserve cluster ordering)
from itertools import permutations

def match_clusters(mu_true, mu_est, K):
    """Find the permutation of estimated clusters that best matches true clusters.
    Uses toroidal distance."""
    best_perm = None
    best_dist = np.inf
    true_centers = np.array([[p[0], p[1]] for p in mu_true])
    
    for perm in permutations(range(K)):
        total_dist = 0
        for k_true, k_est in enumerate(perm):
            for d in range(2):
                diff = np.abs(true_centers[k_true, d] - mu_est[k_est, d])
                diff = min(diff, 2*np.pi - diff)
                total_dist += diff**2
        if total_dist < best_dist:
            best_dist = total_dist
            best_perm = perm
    return list(best_perm)

perm = match_clusters(true_params, mu_em, K)
print(f'Cluster matching (true → estimated): {list(enumerate(perm))}')

# Extract true kappas
true_kappa_theta = [true_params[k][2] for k in range(K)]
true_kappa_phi = [true_params[k][3] for k in range(K)]

# Reorder estimated to match true
est_kappa_theta = [kappa_em[perm[k], 0] for k in range(K)]
est_kappa_phi = [kappa_em[perm[k], 1] for k in range(K)]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

x_pos = np.arange(K)
bar_width = 0.35

# κ_θ comparison
ax = axes[0]
bars1 = ax.bar(x_pos - bar_width/2, true_kappa_theta, bar_width,
               color=[COLORS[k] for k in range(K)], alpha=0.6, label='True', edgecolor='black')
bars2 = ax.bar(x_pos + bar_width/2, est_kappa_theta, bar_width,
               color=[COLORS[k] for k in range(K)], alpha=1.0, label='Estimated', edgecolor='black',
               hatch='//')
ax.set_xlabel('Cluster', fontsize=13)
ax.set_ylabel(r'$\kappa_\theta$', fontsize=14)
ax.set_title(r'Concentration $\kappa_\theta$ — True vs Estimated', fontsize=15)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Cluster {k}' for k in range(K)])
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# κ_φ comparison
ax = axes[1]
bars1 = ax.bar(x_pos - bar_width/2, true_kappa_phi, bar_width,
               color=[COLORS[k] for k in range(K)], alpha=0.6, label='True', edgecolor='black')
bars2 = ax.bar(x_pos + bar_width/2, est_kappa_phi, bar_width,
               color=[COLORS[k] for k in range(K)], alpha=1.0, label='Estimated', edgecolor='black',
               hatch='//')
ax.set_xlabel('Cluster', fontsize=13)
ax.set_ylabel(r'$\kappa_\phi$', fontsize=14)
ax.set_title(r'Concentration $\kappa_\phi$ — True vs Estimated', fontsize=15)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Cluster {k}' for k in range(K)])
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Toroidal K-Means vs von Mises Mixture

Let's compare our probabilistic approach to the hard-assignment toroidal K-means from the previous notebook.

In [ ]:
def toroidal_distance(a, b):
    """Compute toroidal distance between points a and b on [0, 2π)^2."""
    diff = np.abs(a - b)
    diff = np.minimum(diff, 2*np.pi - diff)
    return np.sqrt(np.sum(diff**2, axis=-1))

def circular_mean(angles, weights=None):
    """Compute circular (Fréchet) mean of angles."""
    if weights is None:
        weights = np.ones(len(angles))
    S = np.sum(weights * np.sin(angles))
    C = np.sum(weights * np.cos(angles))
    return np.arctan2(S, C) % (2 * np.pi)

def toroidal_kmeans(data, K, max_iter=100, seed=123):
    """K-means with toroidal distance and circular means."""
    rng = np.random.RandomState(seed)
    N = len(data)
    
    # Initialize with random data points
    idx = rng.choice(N, K, replace=False)
    centroids = data[idx].copy()
    
    for it in range(max_iter):
        # Assign
        dists = np.zeros((N, K))
        for k in range(K):
            dists[:, k] = toroidal_distance(data, centroids[k])
        labels = np.argmin(dists, axis=1)
        
        # Update centroids using circular mean
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            mask = labels == k
            if mask.sum() == 0:
                new_centroids[k] = centroids[k]
            else:
                for d in range(2):
                    new_centroids[k, d] = circular_mean(data[mask, d])
        
        # Check convergence
        shift = toroidal_distance(centroids, new_centroids).sum()
        centroids = new_centroids
        if shift < 1e-8:
            break
    
    return centroids, labels

# Run toroidal K-means
centroids_km, labels_km = toroidal_kmeans(data, K, seed=123)
print('Toroidal K-Means complete.')

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# Left: K-Means
ax = axes[0]
for k in range(K):
    mask = labels_km == k
    ax.scatter(data[mask, 0], data[mask, 1], c=COLORS[k], s=20, alpha=0.6)
    ax.scatter(centroids_km[k, 0], centroids_km[k, 1], c=COLORS[k], s=300,
              marker='*', edgecolors='black', linewidths=1.5, zorder=10)
ax.set_xlim(0, 2*np.pi)
ax.set_ylim(0, 2*np.pi)
ax.set_xlabel(r'$\theta$', fontsize=14)
ax.set_ylabel(r'$\phi$', fontsize=14)
ax.set_title('Toroidal K-Means (Hard Assignments)', fontsize=15)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Right: von Mises EM
ax = axes[1]
# Background posterior regions
bg_image2 = np.zeros((grid_res, grid_res, 4))
for k in range(K):
    rgba = np.array(to_rgba(COLORS[k]))
    mask_g = assignment_grid == k
    bg_image2[mask_g] = rgba
    bg_image2[mask_g, 3] = 0.15 + 0.35 * confidence_grid[mask_g]
ax.imshow(bg_image2, extent=[0, 2*np.pi, 0, 2*np.pi], origin='lower', aspect='equal')

for k in range(K):
    mask = labels_em == k
    ax.scatter(data[mask, 0], data[mask, 1], c=COLORS[k], s=20, alpha=0.7,
              edgecolors='white', linewidths=0.3)
    ax.scatter(mu_em[k, 0], mu_em[k, 1], c=COLORS[k], s=300,
              marker='*', edgecolors='black', linewidths=1.5, zorder=10)
ax.set_xlim(0, 2*np.pi)
ax.set_ylim(0, 2*np.pi)
ax.set_xlabel(r'$\theta$', fontsize=14)
ax.set_ylabel(r'$\phi$', fontsize=14)
ax.set_title('von Mises Mixture EM (Soft Assignments)', fontsize=15)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Rand Index comparison
def rand_index(labels1, labels2):
    """Compute the Rand Index between two clusterings."""
    n = len(labels1)
    a = 0  # pairs in same cluster in both
    b = 0  # pairs in different clusters in both
    for i in range(n):
        for j in range(i+1, n):
            same1 = labels1[i] == labels1[j]
            same2 = labels2[i] == labels2[j]
            if same1 and same2:
                a += 1
            elif not same1 and not same2:
                b += 1
    total_pairs = n * (n - 1) // 2
    return (a + b) / total_pairs

# Compute Rand Index for both methods vs true labels
ri_kmeans = rand_index(labels_true, labels_km)
ri_em = rand_index(labels_true, labels_em)

print(f'Rand Index — Toroidal K-Means vs True: {ri_kmeans:.4f}')
print(f'Rand Index — von Mises EM vs True:     {ri_em:.4f}')

# Bar chart
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
methods = ['Toroidal K-Means', 'von Mises EM']
scores = [ri_kmeans, ri_em]
bar_colors = [COLORS[0], COLORS[2]]

bars = ax.bar(methods, scores, color=bar_colors, edgecolor='black', width=0.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.4f}', ha='center', fontsize=13, fontweight='bold')

ax.set_ylabel('Rand Index', fontsize=13)
ax.set_title('Clustering Quality — Rand Index vs Ground Truth', fontsize=15)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Choosing K — PVE & Silhouette Analysis

How do we know K = 5 is the right number of clusters? We use two metrics:

**PVE (Proportion of Variance Explained)**

$$\text{PVE}(K) = 1 - \frac{\text{TWCSS}_K}{\text{TWCSS}_1}$$

where $\text{TWCSS}_K$ is the **total within-cluster sum of squared distances** for K clusters and $\text{TWCSS}_1$ is the single-cluster baseline (all points assigned to one cluster). PVE ranges from 0 to 1; an "elbow" in the curve marks a natural K choice.

**Silhouette Score**

For each point $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, +1]$$

where $a(i)$ = mean distance to other points in the **same** cluster (cohesion), and $b(i)$ = mean distance to points in the **nearest other** cluster (separation). Computed from the precomputed pairwise distance matrix so it automatically respects this notebook's metric.

In [ ]:
def silhouette_from_matrix(D, labels):
    """
    Mean silhouette score from a precomputed N×N distance matrix.
    s(i) = (b(i) - a(i)) / max(a(i), b(i))
    """
    n = len(labels)
    unique_clusters = np.unique(labels)
    if len(unique_clusters) <= 1:
        return 0.0
    scores = np.zeros(n)
    for i in range(n):
        ci = labels[i]
        same_mask = (labels == ci).copy()
        same_mask[i] = False
        a_i = float(np.mean(D[i, same_mask])) if np.any(same_mask) else 0.0
        b_i = np.inf
        for cj in unique_clusters:
            if cj == ci:
                continue
            other_mask = (labels == cj)
            if np.any(other_mask):
                b_i = min(b_i, float(np.mean(D[i, other_mask])))
        denom = max(a_i, b_i)
        scores[i] = (b_i - a_i) / denom if denom > 1e-15 else 0.0
    return float(np.mean(scores))

In [ ]:
# For the von Mises mixture, use hard assignments (labels_em = argmax of resp_em)
# and toroidal distance for TWCSS and silhouette.
# toroidal_distance and circular_mean are already defined above (see Section 6).

# Build full N×N toroidal pairwise distance matrix
n_pts = len(data)
print("Building pairwise toroidal distance matrix...")
D_full_vm = np.zeros((n_pts, n_pts))
for i in range(n_pts):
    D_full_vm[i] = toroidal_distance(data, data[i])

# TWCSS baseline: one cluster, centred on circular mean of all points
overall_center_vm = np.array([
    circular_mean(data[:, 0]),
    circular_mean(data[:, 1])
])
TWCSS_1 = float(np.sum(toroidal_distance(data, overall_center_vm) ** 2))

K_range = range(2, 11)
pve_vals = []
sil_vals = []

print(f"{'K':>4} {'TWCSS':>10} {'PVE':>8} {'Silhouette':>12}")
print("-" * 38)
for k in K_range:
    mu_k, _, _, resp_k, _ = toroidal_vm_em(data, k, max_iter=100, seed=123)
    lbls_k = np.argmax(resp_k, axis=1)
    # TWCSS: distance from each point to its assigned component mean
    dists_k = toroidal_distance(data, mu_k[lbls_k])
    twcss_k = float(np.sum(dists_k ** 2))
    pve = 1 - twcss_k / TWCSS_1
    sil = silhouette_from_matrix(D_full_vm, lbls_k)
    pve_vals.append(pve)
    sil_vals.append(sil)
    print(f"{k:>4} {twcss_k:>10.4f} {pve:>8.4f} {sil:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
K_chosen = 5
K_vals = list(K_range)

ax = axes[0]
ax.plot(K_vals, pve_vals, 'o-', color='#2c3e50', linewidth=2, markersize=7,
        markerfacecolor='#3498db', markeredgecolor='white', markeredgewidth=1.5)
ax.axvline(K_chosen, color='#e74c3c', linestyle='--', linewidth=1.8,
           label=f'Chosen K = {K_chosen}')
ax.fill_between(K_vals, pve_vals, alpha=0.08, color='#3498db')
ax.set_xlabel('Number of Clusters K', fontsize=12)
ax.set_ylabel('PVE', fontsize=12)
ax.set_title('PVE Elbow Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_xticks(K_vals); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(K_vals, sil_vals, 's-', color='#2c3e50', linewidth=2, markersize=7,
        markerfacecolor='#9b59b6', markeredgecolor='white', markeredgewidth=1.5)
best_k_sil = K_vals[int(np.argmax(sil_vals))]
ax.axvline(best_k_sil, color='#9b59b6', linestyle='--', linewidth=1.8,
           label=f'Best silhouette K = {best_k_sil}')
ax.axvline(K_chosen, color='#e74c3c', linestyle=':', linewidth=1.8,
           label=f'Chosen K = {K_chosen}')
ax.fill_between(K_vals, sil_vals, alpha=0.08, color='#9b59b6')
ax.set_xlabel('Number of Clusters K', fontsize=12)
ax.set_ylabel('Mean Silhouette Score', fontsize=12)
ax.set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_xticks(K_vals); ax.grid(True, alpha=0.3)

plt.suptitle('Choosing K: PVE & Silhouette Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nK=2..10 summary:")
print(f"{'K':>4} {'PVE':>8} {'Silhouette':>12}")
print("-" * 27)
for k, pve, sil in zip(K_vals, pve_vals, sil_vals):
    marker = " ◄ chosen" if k == K_chosen else ""
    print(f"{k:>4} {pve:>8.4f} {sil:>12.4f}{marker}")

## 7. Summary

| Aspect | Toroidal K-Means | von Mises Mixture (EM) |
|--------|-----------------|------------------------|
| **Assignment** | Hard (each point → 1 cluster) | Soft (probability per cluster) |
| **Distribution** | None assumed | von Mises per dimension |
| **Parameters per cluster** | Center only ($\mu_\theta, \mu_\phi$) | Center + concentration ($\mu_\theta, \mu_\phi, \kappa_\theta, \kappa_\phi$) + weight $\pi_k$ |
| **Handles varying spread** | No — all clusters same "size" | Yes — $\kappa$ adapts per cluster per dimension |
| **Wraparound handling** | Toroidal distance + circular mean | Built-in via $\cos(\theta - \mu)$ periodicity |
| **Best for** | Quick exploration, spherical-ish clusters | Clusters with different spreads, uncertainty quantification |

### Key Takeaways

1. **The von Mises distribution is the natural "Gaussian" for circular data.** Its density $\exp(\kappa\cos(\theta-\mu))$ is inherently periodic, so wraparound is handled automatically — no special distance functions needed.

2. **Soft assignments reveal uncertainty.** Points near cluster boundaries get split responsibility, giving us calibrated confidence rather than forced hard decisions.

3. **Per-cluster, per-dimension concentrations ($\kappa_{k\theta}, \kappa_{k\phi}$)** let the model adapt to clusters of different tightness — something K-means fundamentally cannot do.

4. **Numerical stability matters.** Using `i0e` (exponentially-scaled Bessel function) and log-sum-exp prevents overflow/underflow that would plague naive implementations with large $\kappa$.

### Next: Mixed-Type — Gaussian + Categorical Mixture